# 📱 ConnectaTel: Análisis de Uso y Segmentación de Clientes

Como **analista de datos**, el objetivo es evaluar el **comportamiento de los clientes** de ConnectaTel, una empresa de telecomunicaciones con operaciones en **México y Colombia**.

Se trabaja con información registrada **hasta el año 2024**, lo cual permite analizar el comportamiento del negocio dentro de ese periodo.

Para ello se utilizan tres datasets:

- **plans.csv** → información de los planes actuales (precio, minutos incluidos, GB incluidos, costo por extra)
- **users_latam.csv** → información de los clientes (edad, ciudad, fecha de registro, plan, churn)
- **usage.csv** → detalle del **uso real** de los servicios (llamadas y mensajes)

El análisis consiste en **explorar**, **limpiar** y **analizar** estos datos para construir un **perfil estadístico** de los clientes, detectar **comportamientos atípicos** y crear **segmentos de clientes**, con el fin de **identificar patrones de consumo**, **diseñar estrategias de retención** y **sugerir mejoras en los planes** ofrecidos por la empresa.

---
## 🧩 Paso 1: Cargar y explorar

Antes de limpiar o combinar los datos, es necesario **familiarizarse con la estructura de los tres datasets**: validar que los archivos se carguen correctamente, conocer sus columnas y tipos de datos, y detectar posibles inconsistencias.

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:**
Tener los **3 datasets listos en memoria**, entender su contenido y realizar una revisión preliminar.

**Instrucciones:**
- Importa las librerías necesarias (`pandas`, `numpy`, `seaborn`, `matplotlib.pyplot`).
- Carga los archivos CSV usando `pd.read_csv()`: **`plans.csv`**, **`users_latam.csv`** y **`usage.csv`**.
- Guarda los DataFrames en las variables: `plans`, `users`, `usage`.
- Muestra las primeras filas de cada DataFrame usando `.head()`.

In [ ]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# cargar archivos
import os

# En la plataforma del curso los archivos están en '/datasets/'; localmente, en la misma carpeta del notebook
ruta = '/datasets/' if os.path.exists('/datasets/') else ''

plans = pd.read_csv(ruta + 'plans.csv')
users = pd.read_csv(ruta + 'users_latam.csv')
usage = pd.read_csv(ruta + 'usage.csv')

In [ ]:
# mostrar las primeras 5 filas de plans
plans.head()

In [ ]:
# mostrar las primeras 5 filas de users
users.head()

In [ ]:
# mostrar las primeras 5 filas de usage
usage.head()

### 1.2 Exploración de la estructura de los datasets

**🎯 Objetivo:**
Conocer la **estructura de cada dataset**, revisar cuántas filas y columnas tienen, identificar los **tipos de datos** de cada columna y detectar posibles **inconsistencias o valores nulos** antes de iniciar el análisis.

**Instrucciones:**
- Revisa el **número de filas y columnas** de cada dataset usando `.shape`.
- Usa `.info()` en cada DataFrame para obtener un **resumen completo** de columnas, tipos de datos y valores no nulos.

In [ ]:
# revisar el número de filas y columnas de cada dataset
print("plans", plans.shape)
print("users", users.shape)
print("usage", usage.shape)

In [ ]:
# inspección de plans con .info()
plans.info()

In [ ]:
# inspección de users con .info()
users.info()

In [ ]:
# inspección de usage con .info()
usage.info()

---
## 🧩 Paso 2: Identificación de problemas de calidad de datos

### 2.1 Revisión de valores nulos

**🎯 Objetivo:**
Detectar la presencia y magnitud de valores faltantes para evaluar si afectan el análisis o requieren imputación/eliminación.

**Instrucciones:**
- Cuenta valores nulos por columna para cada dataset.
- Calcula la proporción de nulos por columna para cada dataset.

El dataset `plans` solamente tiene 2 renglones y no tiene ausentes, por ello no necesita exploración adicional.

In [ ]:
# cantidad de nulos para users
print(users.isna().sum())                       # Cantidad de valores nulos
print((users.isna().mean() * 100).round(2))     # Proporción de valores nulos (%)

In [ ]:
# cantidad de nulos para usage
print(usage.isna().sum())                       # Cantidad de valores nulos
print((usage.isna().mean() * 100).round(2))     # Proporción de valores nulos (%)

✍️ **Diagnóstico: valores nulos**

**¿Qué columnas tienen valores faltantes, en qué proporción y qué haría con ellas?**

**users**
- `city` tiene 469 nulos (11.72%). Como está entre 5% y 30%, se dejan como nulos, porque no se puede adivinar la ciudad de un cliente.
- `churn_date` tiene 3,534 nulos (88.35%). No se elimina, porque el nulo significa que el cliente no se ha dado de baja (sigue activo).
- El resto de las columnas no tiene nulos.

**usage**
- `date` tiene 50 nulos (0.12%). Como es menos del 5%, se dejan como nulos.
- `duration` tiene 55.19% de nulos y `length` 44.74%. No se imputan, porque los mensajes no tienen duración y las llamadas no tienen longitud (se verifica en el Paso 3.2).

### 2.2 Detección de valores inválidos y sentinels

**🎯 Objetivo:**
Identificar sentinels: valores que no deberían estar en el dataset.

**Instrucciones:**
- Explora las columnas numéricas con **un resumen estadístico** y describe brevemente qué encontraste.
- Explora las columnas categóricas **relevantes**, revisando sus valores únicos, y describe brevemente qué encontraste.

El dataset `plans` solamente tiene 2 renglones, por ello no necesita exploración adicional.

In [ ]:
# explorar columnas numéricas de users
users.describe()

- La columna `user_id` va de 10000 a 13999 y tiene 4,000 registros, uno por cliente. No tiene valores raros, es solo un identificador.
- La columna `age` tiene un valor mínimo de **-999**, que es un sentinel porque no existe una edad negativa. Este valor hace que la media (33.7) baje y que la desviación estándar (123.2) sea muy alta. El resto de las edades llega hasta 79 años.

In [ ]:
# explorar columnas numéricas de usage
usage.describe()

- Las columnas `id` y `user_id` son identificadores. `id` va de 1 a 40000, uno por cada registro de uso, y `user_id` va de 10000 a 13999, el mismo rango que en `users`. No tienen valores raros.
- Las columnas `duration` y `length` no tienen valores negativos ni sentinels. `duration` va de 0 a 120 minutos y `length` de 0 a 1,490 caracteres. En ambas el máximo está muy lejos del percentil 75 (6.99 y 64), así que podría haber valores atípicos que se revisarán más adelante.

In [ ]:
# explorar columnas categóricas de users
columnas_user = ['city', 'plan']

for col in columnas_user:
    print(f'--- {col} ---')
    print(users[col].value_counts(dropna=False))
    print()

- La columna `city` tiene 6 ciudades: CDMX, GDL y MTY (México) y Bogotá, Medellín y Cali (Colombia). Además tiene 469 nulos y 96 registros con **"?"**, que es un sentinel porque no es una ciudad válida.
- La columna `plan` solo tiene dos valores, "Basico" (2,595) y "Premium" (1,405), que coinciden con los planes del dataset `plans`. No tiene valores inválidos.

In [ ]:
# explorar columna categórica de usage
usage['type'].value_counts(dropna=False)

- La columna `type` solo tiene dos valores: "text" (22,092) y "call" (17,908). No tiene nulos ni valores inválidos. Esto explica los nulos de `duration` y `length`, porque los mensajes no tienen duración y las llamadas no tienen longitud.

✍️ **Diagnóstico: valores inválidos o sentinels**

**¿En qué columnas se encontraron valores inválidos o sentinels?**
- `age`: tiene el valor **-999**, que no es una edad válida.
- `city`: tiene el valor **"?"** en 96 registros, que no es una ciudad válida.
- En `usage` no se encontraron sentinels: `type` solo tiene "text" y "call", y `duration` y `length` no tienen valores negativos.

**¿Qué acción tomaría?**
- Reemplazar -999 en `age` por la mediana de las edades válidas, y "?" en `city` por valores nulos, para que no afecten los cálculos y se traten igual que los demás datos faltantes.

### 2.3 Revisión y estandarización de fechas

**🎯 Objetivo:**
Asegurar que las columnas de fecha estén correctamente formateadas y detectar años fuera de rango que indiquen errores de captura.

**Instrucciones:**
- Convierte las columnas de fecha a tipo fecha y asegúrate de que el código sea a prueba de errores.
- Revisa cuántas veces aparece cada año.
- Identifica fechas imposibles (ej. años futuros o negativos).

Se toma en cuenta que hay datos registrados hasta el año 2024.

In [ ]:
# Convertir a fecha la columna `reg_date` de users
users['reg_date'] = pd.to_datetime(users['reg_date'], errors='coerce')

In [ ]:
# Convertir a fecha la columna `date` de usage
usage['date'] = pd.to_datetime(usage['date'], errors='coerce')

In [ ]:
# Revisar los años presentes en `reg_date` de users
users['reg_date'].dt.year.value_counts(dropna=False).sort_index()

En `reg_date`, los registros están repartidos de forma similar entre 2022 (1,314), 2023 (1,316) y 2024 (1,330), pero hay **40 registros con año 2026**, un año que todavía no había pasado cuando se guardaron los datos (los datos llegan hasta 2024). No hay fechas nulas.

In [ ]:
# Revisar los años presentes en `date` de usage
usage['date'].dt.year.value_counts(dropna=False).sort_index()

En `date`, todos los registros son del año 2024 (39,950) y hay 50 nulos, que son los mismos 50 vistos en la revisión de nulos. No hay años fuera de rango.
Se basará el análisis en estas fechas.

✍️ **Diagnóstico: fechas fuera de rango**

**¿Aparecen años imposibles?**
- Sí, en `reg_date` hay 40 registros con año 2026, que es imposible porque los datos llegan hasta 2024. En `date` no hay años imposibles.

**¿Qué haría con ellas?**
- Reemplazar esas 40 fechas por nulos (NaT) en lugar de eliminar a los usuarios, porque el resto de su información (edad, ciudad, plan) sí es válida.

---
## 🧩 Paso 3: Limpieza básica de datos

### 3.1 Corregir sentinels y fechas imposibles

**🎯 Objetivo:**
Aplicar reglas de limpieza para reemplazar valores sentinels y corregir fechas imposibles.

**Instrucciones:**
- En `age`, reemplaza el sentinel **-999** con la mediana.
- En `city`, reemplaza el sentinel `"?"` por valores nulos (`pd.NA`).
- Marca como nulas las fechas fuera de rango.

In [ ]:
# Reemplazar -999 por la mediana de age (calculada solo con edades válidas)
age_mediana = users.loc[users['age'] != -999, 'age'].median()
users['age'] = users['age'].replace(-999, age_mediana)

# Verificar cambios
users['age'].describe()

In [ ]:
# Reemplazar ? por NA en city
users['city'] = users['city'].replace('?', pd.NA)

# Verificar cambios
users['city'].value_counts(dropna=False)

In [ ]:
# Marcar fechas futuras como nulas para reg_date (pd.NaT es el valor nulo de las columnas de fecha)
users.loc[users['reg_date'].dt.year > 2024, 'reg_date'] = pd.NaT

# Verificar cambios
users['reg_date'].dt.year.value_counts(dropna=False).sort_index()

### 3.2 Tratamiento de valores nulos

**🎯 Objetivo:**
Decidir qué hacer con los valores nulos según su proporción y relevancia.

**Instrucciones:**
- Verifica si los nulos en `duration` y `length` son **MAR** (Missing At Random) revisando si dependen de la columna `type`.
  Si se confirma que son MAR, **se dejan como nulos** y se justifica la decisión.

In [ ]:
# Verificación MAR en usage (Missing At Random) para duration
usage['duration'].isna().groupby(usage['type']).mean() * 100

In [ ]:
# Verificación MAR en usage (Missing At Random) para length
usage['length'].isna().groupby(usage['type']).mean() * 100

Los nulos en `duration` y `length` dependen de la columna `type`:
- En `duration`, el **99.93%** de los registros de tipo "text" son nulos, contra **0%** en "call", porque los mensajes no tienen duración.
- En `length`, el **99.93%** de los registros de tipo "call" son nulos, contra **0%** en "text", porque las llamadas no tienen longitud de texto.

Por lo tanto, los nulos son **MAR** (Missing At Random): no son errores de captura, sino que se explican por el tipo de evento. La decisión es **dejarlos como nulos**, porque imputarlos significaría inventar una duración a los mensajes o una longitud a las llamadas, lo cual no tiene sentido y sesgaría el análisis.

---
## 🧩 Paso 4: Summary statistics de uso por usuario

### 4.1 Agrupación por comportamiento de uso

**🎯 Objetivo:** Resumir las variables clave de la tabla `usage` **por usuario**, creando métricas que representen su comportamiento real de uso histórico.

**Instrucciones:**
1. Construye una tabla agregada de `usage` por `user_id` que incluya:
   - número total de mensajes
   - número total de llamadas
   - total de minutos de llamadas
2. Renombra las columnas para que tengan nombres claros: `cant_mensajes`, `cant_llamadas` y `cant_minutos_llamada`.
3. Combina esta tabla con `users`.

In [ ]:
# Columnas auxiliares
usage["is_text"] = (usage["type"] == "text").astype(int) # conocer el total de mensajes
usage["is_call"] = (usage["type"] == "call").astype(int) # conocer el total de llamadas

# Agrupar información por usuario
usage_agg = usage.groupby('user_id').agg({
    'is_text': 'sum',
    'is_call': 'sum',
    'duration': 'sum'
}).reset_index()

# observar resultado
usage_agg.head(3)

In [ ]:
# Renombrar columnas
usage_agg = usage_agg.rename(columns={
    'is_text': 'cant_mensajes',
    'is_call': 'cant_llamadas',
    'duration': 'cant_minutos_llamada'
})

# observar resultado
usage_agg.head(3)

In [ ]:
# Combinar la tabla agregada con el dataset de usuarios
user_profile = users.merge(usage_agg, on='user_id', how='left')

# Un usuario no tiene registros de uso: sus métricas se completan con 0
columnas_uso = ['cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']
user_profile[columnas_uso] = user_profile[columnas_uso].fillna(0)

user_profile.head(5)

### 4.2 Resumen estadístico por usuario durante el 2024

**🎯 Objetivo:** Analizar las columnas numéricas y categóricas de los usuarios para identificar rangos, valores extremos y distribución de los datos antes de continuar con el análisis.

**Instrucciones:**
1. Para las columnas **numéricas** relevantes, obtén un resumen estadístico (media, mediana, mínimo, máximo, etc.).
2. Para la columna **categórica** `plan`, revisa la distribución en **porcentajes** de cada categoría.

In [ ]:
# Resumen estadístico de las columnas numéricas
user_profile[['age', 'cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']].describe()

In [ ]:
# Distribución porcentual del tipo de plan
(user_profile['plan'].value_counts(normalize=True) * 100).round(2)

---
## 🧩 Paso 5: Visualización de distribuciones (uso y clientes) y outliers

### 5.1 Visualización de distribuciones

**🎯 Objetivo:**
Entender visualmente cómo se comportan las variables clave tanto de **uso** como de **clientes**, observar si existen diferencias según el tipo de plan, y analizar la **forma de la distribución**.

**Instrucciones:**
Graficar **histogramas** para las columnas `age`, `cant_mensajes`, `cant_llamadas` y `cant_minutos_llamada`, y después de cada gráfico escribir un **insight** respecto al plan, la variable y el tipo de distribución (simétrica, sesgada a la derecha o a la izquierda).

Para cada histograma se usa `hue='plan'` para comparar Básico y Premium, la paleta `['skyblue', 'green']`, y título y etiquetas.

In [ ]:
# Histograma para visualizar la edad (age)
sns.histplot(data=user_profile, x='age', hue='plan', palette=['skyblue', 'green'])
plt.title('Distribución de la edad de los usuarios por plan')
plt.xlabel('Edad')
plt.ylabel('Cantidad de usuarios')
plt.show()

💡 **Insights:**
- **Distribución:** la edad es aproximadamente simétrica y bastante plana: los usuarios están repartidos de forma pareja entre los 18 y los 79 años, con una mediana de 48 años. Se ve un pico en el rango de 46 a 50 años, causado por los 55 valores -999 que se reemplazaron con la mediana en la limpieza.
- **Plan:** el plan Básico tiene más usuarios en todos los rangos de edad, porque concentra al 65% de los clientes. La proporción de usuarios Premium es parecida en todas las edades, así que no existe un patrón claro entre la edad y el tipo de plan.

In [ ]:
# Histograma para visualizar la cant_mensajes
sns.histplot(data=user_profile, x='cant_mensajes', hue='plan', palette=['skyblue', 'green'])
plt.title('Distribución de la cantidad de mensajes por plan')
plt.xlabel('Cantidad de mensajes')
plt.ylabel('Cantidad de usuarios')
plt.show()

💡 **Insights:**
- **Distribución:** la cantidad de mensajes por usuario tiene un ligero sesgo a la derecha; la mayoría envía entre 4 y 7 mensajes (mediana de 5) y unos cuantos llegan hasta 17.
- **Plan:** ambos planes tienen la misma forma de distribución, con el pico entre 4 y 6 mensajes. Los usuarios Premium no envían más mensajes que los Básico: no existe un patrón que los distinga.

In [ ]:
# Histograma para visualizar la cant_llamadas
sns.histplot(data=user_profile, x='cant_llamadas', hue='plan', palette=['skyblue', 'green'])
plt.title('Distribución de la cantidad de llamadas por plan')
plt.xlabel('Cantidad de llamadas')
plt.ylabel('Cantidad de usuarios')
plt.show()

💡 **Insights:**
- **Distribución:** la cantidad de llamadas por usuario tiene un ligero sesgo a la derecha; la mayoría hace entre 3 y 6 llamadas (mediana de 4) y pocos usuarios llegan hasta 15.
- **Plan:** los dos planes se comportan igual, con el pico entre 3 y 4 llamadas. Los usuarios Premium no hacen más llamadas que los Básico: no existe un patrón que los distinga.

In [ ]:
# Histograma para visualizar la cant_minutos_llamada
sns.histplot(data=user_profile, x='cant_minutos_llamada', hue='plan', palette=['skyblue', 'green'])
plt.title('Distribución de los minutos de llamada por plan')
plt.xlabel('Minutos de llamada')
plt.ylabel('Cantidad de usuarios')
plt.show()

💡 **Insights:**
- **Distribución:** los minutos de llamada por usuario tienen un sesgo a la derecha más marcado que las otras variables: la mediana es de unos 20 minutos, pero algunos usuarios superan los 150, porque hay llamadas individuales de hasta 120 minutos.
- **Plan:** ambos planes tienen la misma forma, con la mayoría de los usuarios entre 5 y 40 minutos. Además, se observa un pequeño grupo separado de usuarios de ambos planes con entre 130 y 155 minutos, muy lejos del resto.

### 5.2 Identificación de outliers

**🎯 Objetivo:**
Detectar valores extremos en las variables clave de **uso** y **clientes** que podrían afectar el análisis, y decidir si requieren limpieza o revisión adicional.

**Instrucciones:**
- Usa **boxplots** para identificar visualmente outliers en `age`, `cant_mensajes`, `cant_llamadas` y `cant_minutos_llamada`, generándolos automáticamente con un **for**.
- Después de crear los gráficos, responde si **existen o no outliers** en las variables.
- Si hay outliers, crea otro bucle para calcular los límites de esas columnas usando el **método IQR** y decide qué hacer con ellos. Si solamente hay outliers de un solo lado, no es necesario calcular ambos límites.

In [ ]:
# Visualizando usando BoxPlot
columnas_numericas = ['age', 'cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']

for col in columnas_numericas:
    sns.boxplot(x=user_profile[col])
    plt.title(f'Boxplot: {col}')
    plt.xlabel(col)
    plt.show()

💡 **Insights:**
- **age:** no presenta outliers; todas las edades están dentro del rango normal del boxplot.
- **cant_mensajes:** sí presenta outliers, solo del lado superior.
- **cant_llamadas:** sí presenta outliers, solo del lado superior.
- **cant_minutos_llamada:** sí presenta outliers, solo del lado superior, y es la variable con más valores extremos.

Como los outliers aparecen únicamente del lado superior (no puede haber valores menores a 0), solo se calcula el límite superior.

In [ ]:
# Calcular límites con el método IQR
columnas_limites = ['cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']

for col in columnas_limites:
    Q1 = user_profile[col].quantile(0.25)
    Q3 = user_profile[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_superior = Q3 + 1.5 * IQR
    cant_outliers = (user_profile[col] > limite_superior).sum()
    print(f'{col}: límite superior = {limite_superior:.2f}, outliers = {cant_outliers}')

In [ ]:
# Revisa los límites superiores y el max, para tomar la decisión de mantener los outliers o no
user_profile[columnas_limites].describe()

💡 **Insights: ¿mantener o no los outliers?**
- **cant_mensajes:** se **mantienen**. 46 usuarios superan el límite superior de 11.5 mensajes y el máximo es 17; son valores posibles de clientes que envían muchos mensajes, no errores de registro.
- **cant_llamadas:** se **mantienen**. 30 usuarios superan el límite de 10.5 llamadas y el máximo es 15; corresponden a clientes reales con alto uso.
- **cant_minutos_llamada:** se **mantienen**. 109 usuarios superan el límite de aproximadamente 62 minutos y el máximo es de unos 156; son valores posibles (hay llamadas reales de hasta 120 minutos) y representan a los clientes que más usan el servicio.

---
## 🧩 Paso 6: Segmentación de clientes

### 6.1 Segmentación de clientes por uso

**🎯 Objetivo:** Clasificar a cada usuario en un grupo de uso (Bajo uso, Uso medio, Alto uso) basándose en la cantidad de llamadas y mensajes registrados.

**Instrucciones:**
- Crea una nueva columna llamada `grupo_uso` en el dataframe `user_profile`.
- Usa comparaciones lógicas (<, >) para evaluar las condiciones de llamadas y mensajes y asigna:
  - `'Bajo uso'` cuando llamadas < 5 y mensajes < 5
  - `'Uso medio'` cuando llamadas < 10 y mensajes < 10
  - `'Alto uso'` para el resto de casos

In [ ]:
# Crear columna grupo_uso
def clasificar_uso(row):
    if row['cant_llamadas'] < 5 and row['cant_mensajes'] < 5:
        return 'Bajo uso'
    elif row['cant_llamadas'] < 10 and row['cant_mensajes'] < 10:
        return 'Uso medio'
    else:
        return 'Alto uso'

user_profile['grupo_uso'] = user_profile.apply(clasificar_uso, axis=1)

In [ ]:
# verificar cambios
user_profile.head()

### 6.2 Segmentación de clientes por edad

**🎯 Objetivo:** Clasificar a cada usuario en un grupo por **edad**.

**Instrucciones:**
- Crea una nueva columna llamada `grupo_edad` en el dataframe `user_profile`.
- Usa comparaciones lógicas (<, >) para evaluar las condiciones y asigna:
  - `'Joven'` cuando age < 30
  - `'Adulto'` cuando age < 60
  - `'Adulto Mayor'` para el resto de casos

In [ ]:
# Crear columna grupo_edad
def clasificar_edad(age):
    if age < 30:
        return 'Joven'
    elif age < 60:
        return 'Adulto'
    else:
        return 'Adulto Mayor'

user_profile['grupo_edad'] = user_profile['age'].apply(clasificar_edad)

In [ ]:
# verificar cambios
user_profile.head()

### 6.3 Visualización de la segmentación de clientes

**🎯 Objetivo:** Visualizar la distribución de los usuarios según los grupos creados: **grupo_uso** y **grupo_edad**.

**Instrucciones:**
- Crea dos gráficos para las variables categóricas `grupo_uso` y `grupo_edad`.
- Agrega título y etiquetas a los ejes en cada gráfico.

In [ ]:
# Visualización de los segmentos por uso
sns.countplot(data=user_profile, x='grupo_uso', order=['Bajo uso', 'Uso medio', 'Alto uso'])
plt.title('Cantidad de usuarios por grupo de uso')
plt.xlabel('Grupo de uso')
plt.ylabel('Cantidad de usuarios')
plt.show()

In [ ]:
# Visualización de los segmentos por edad
sns.countplot(data=user_profile, x='grupo_edad', order=['Joven', 'Adulto', 'Adulto Mayor'])
plt.title('Cantidad de usuarios por grupo de edad')
plt.xlabel('Grupo de edad')
plt.ylabel('Cantidad de usuarios')
plt.show()

💡 **Insights:**
- **Por uso:** la gran mayoría de los clientes es de **Uso medio** (aprox. 74%), seguido por **Bajo uso** (aprox. 19%); solo alrededor del **7%** es de **Alto uso**.
- **Por edad:** el grupo **Adulto** (30 a 59 años) es el más grande, con aproximadamente la mitad de los clientes, seguido por **Adulto Mayor** (aprox. 31%) y **Joven** (aprox. 19%).

---
## 🧩 Paso 7: Insight ejecutivo para stakeholders

**🎯 Objetivo:** Traducir los hallazgos del análisis en conclusiones accionables para el negocio, enfocadas en segmentación, patrones de uso y oportunidades comerciales.

**Preguntas a responder:**
- ¿Qué problemas tenían originalmente los datos? ¿Qué porcentaje, o cantidad de filas, de esa columna representaban?
- ¿Qué segmentos de clientes se identificaron y cómo se comportan según su edad y nivel de uso?
- ¿Qué segmentos parecen más valiosos para ConnectaTel y por qué?
- ¿Qué patrones de uso extremo (outliers) se encontraron y qué implican para el negocio?
- ¿Qué recomendaciones se harían para mejorar la oferta actual de planes o crear nuevos planes basados en los segmentos y patrones detectados?

## 📋 Análisis ejecutivo

### ⚠️ Problemas detectados en los datos

- **`age`**: tenía el sentinel **-999** en **55 registros (1.4%)**, que no es una edad válida. Se reemplazó por la mediana de las edades válidas (48 años).
- **`city`**: tenía **469 nulos** y **96 registros con "?"**, en total **565 usuarios (14.1%)** sin ciudad válida. Se dejaron como nulos porque no se puede adivinar la ciudad.
- **`reg_date`**: **40 registros (1%)** tenían año **2026**, imposible porque los datos llegan hasta 2024. Se marcaron como nulos.
- **`date`** (usage): **50 nulos (0.12%)**. Se dejaron como nulos por ser una proporción muy baja.
- **`duration` y `length`**: **55.19%** y **44.74%** de nulos, pero son **MAR**: los mensajes no tienen duración y las llamadas no tienen longitud. No son errores y se dejaron como nulos.
- **`churn_date`**: **88.35%** de nulos, que no es un error: indica que **3,534 clientes siguen activos**, mientras que **466 (11.65%)** se dieron de baja.
- **Uso por cliente**: **1 usuario** no tenía registros de uso; sus métricas se completaron con 0 para que no quedara mal clasificado.

### 🔍 Segmentos por edad

- La edad de los clientes está distribuida de forma pareja, sin outliers, con una mediana de **48 años**.
- El grupo **Adulto (30 a 59 años)** es el más grande, con aproximadamente **50%** de los clientes, seguido por **Adulto Mayor (31%)** y **Joven (19%)**.

### 📊 Segmentos por nivel de uso

- La gran mayoría de los clientes es de **Uso medio (aprox. 74%)**, seguido por **Bajo uso (19%)**, y solo un **7%** es de **Alto uso**.
- En total se registraron **22,092 mensajes (55.2%)** y **17,908 llamadas (44.8%)**, por lo que los clientes usan más los mensajes que las llamadas.
- Las variables de uso tienen **sesgo a la derecha** y presentan **outliers** (46 en mensajes, 30 en llamadas y 109 en minutos): un grupo pequeño de clientes usa el servicio mucho más que el resto. Se mantuvieron porque son clientes reales, no errores.
- El plan **Básico** concentra al **64.88%** de los clientes y el **Premium** al **35.12%**.
- Los usuarios **Premium no usan más el servicio que los Básico**: la distribución de mensajes, llamadas y minutos es igual en ambos planes, lo que indica que el plan contratado no refleja el consumo real de los clientes.
- El segmento más valioso para ConnectaTel es el de **Alto uso**, porque son los clientes que más consumen y los mejores candidatos para planes de mayor precio; le sigue el de **Uso medio**, por ser el grupo más grande de clientes.

➡️ **Esto sugiere que** el cliente típico de ConnectaTel es un adulto de uso medio en el plan Básico, mientras que existe un grupo pequeño de usuarios de alto uso (7%) que consume mucho más que el promedio y que representa la mayor oportunidad para ofrecer planes de mayor valor.

### 💡 Recomendaciones

- **Migrar a los usuarios de alto uso al plan Premium**: son el segmento más valioso y los mejores candidatos para una oferta de mejora de plan.
- **Revisar el valor del plan Premium**: como sus usuarios consumen lo mismo que los del Básico, conviene reforzar sus beneficios o ajustar su precio para evitar que lo abandonen.
- **Enfocar las campañas principales en el segmento Adulto de uso medio**, porque es el grupo más grande de clientes.
- **Crear un plan o paquete enfocado en mensajes**, ya que es el servicio más utilizado.
- **Diseñar un plan económico para el segmento de bajo uso (19%)**, para evitar que se den de baja por pagar más de lo que usan (el 11.65% de los clientes ya hizo churn).
- **Mejorar la captura de datos**: validar la edad, la ciudad y las fechas de registro al dar de alta a un cliente, para evitar sentinels y fechas imposibles.

---
## 🧩 Paso 8: Publicación en GitHub

El notebook, los datasets y el README con la descripción del proyecto y la guía de reproducción se encuentran en el repositorio público:

🔗 [Data-Analytics-Portfolio / 06_Python_EDA_Telecom](https://github.com/rcandia-analytics/Data-Analytics-Portfolio/tree/main/06_Python_EDA_Telecom)